# EDA 004.02 — Bivariate Analysis

**Exploring relationships between two variables**

## Learning Objectives
By the end of this notebook you will be able to:
- Identify the correct plot for each pair type (Num×Num, Num×Cat, Cat×Cat)
- Create scatter plots, line charts, grouped box plots, and bar charts
- Visualise associations between two categorical variables
- Add regression lines and confidence intervals to scatter plots

---

## Types of Bivariate Relationships

| Pair type | Best plots |
|---|---|
| Numerical × Numerical | Scatter plot, Line chart, Hexbin, 2D-KDE |
| Numerical × Categorical | Box plot, Violin plot, Bar chart (mean + CI), Strip plot |
| Categorical × Categorical | Grouped/stacked bar chart, Heatmap (cross-tab) |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted')
%matplotlib inline

tips = sns.load_dataset('tips')
titanic = sns.load_dataset('titanic')

print('Tips:', tips.shape)
print('Titanic:', titanic.shape)
tips.head()

---
## 1 — Numerical × Numerical: Scatter Plot

**Reference:** [seaborn.scatterplot](https://seaborn.pydata.org/generated/seaborn.scatterplot.html) | [seaborn.regplot](https://seaborn.pydata.org/generated/seaborn.regplot.html)

Scatter plots reveal:
- **Direction** of association (positive / negative)
- **Strength** of association (tight cloud vs scattered points)
- **Form** (linear, curved, no pattern)
- **Outliers** (isolated points far from the main cloud)

Add a regression line with `regplot` or `lmplot` to estimate the linear trend.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Basic scatter
sns.scatterplot(data=tips, x='total_bill', y='tip', ax=axes[0], alpha=0.6)
axes[0].set_title('Total Bill vs Tip')

# Scatter + regression line + 95% CI band + hue encoding
sns.scatterplot(data=tips, x='total_bill', y='tip',
                hue='time', style='smoker', ax=axes[1], alpha=0.7)
# Add regression line separately
m, b = np.polyfit(tips['total_bill'], tips['tip'], 1)
x_line = np.linspace(tips['total_bill'].min(), tips['total_bill'].max(), 100)
axes[1].plot(x_line, m * x_line + b, 'k--', linewidth=1.5, label='OLS line')
axes[1].set_title('Bill vs Tip (hue=time, style=smoker)')
axes[1].legend()

plt.tight_layout()
plt.show()

corr = tips['total_bill'].corr(tips['tip'])
print(f'Pearson r (bill vs tip): {corr:.3f}')

---
## 2 — Numerical × Numerical: 2D Density (Hexbin / KDE)

**Reference:** [seaborn.kdeplot (2D)](https://seaborn.pydata.org/generated/seaborn.kdeplot.html)

When you have **many overlapping points**, a 2D density plot reveals the true data distribution better than a scatter plot.

In [ ]:
diamonds = sns.load_dataset('diamonds').sample(2000, random_state=42)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Overplotted scatter
axes[0].scatter(diamonds['carat'], diamonds['price'], alpha=0.05, s=5)
axes[0].set_title('Scatter — overplotted (2000 points)')
axes[0].set_xlabel('Carat')
axes[0].set_ylabel('Price')

# 2D KDE with filled contours
sns.kdeplot(data=diamonds, x='carat', y='price', fill=True,
            cmap='Blues', thresh=0.05, ax=axes[1])
axes[1].set_title('2D KDE — shows where density is highest')

plt.tight_layout()
plt.show()

---
## 3 — Numerical × Numerical: Line Chart

**Reference:** [seaborn.lineplot](https://seaborn.pydata.org/generated/seaborn.lineplot.html)

Line charts are ideal when the x-axis is **ordered** (time, sequence) and you want to visualise a trend or trajectory.
Seaborn's `lineplot` automatically aggregates and shows a **confidence interval** band.

In [ ]:
flights = sns.load_dataset('flights')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Aggregate yearly trend with CI across months
sns.lineplot(data=flights, x='year', y='passengers', ax=axes[0])
axes[0].set_title('Annual Passenger Trend (mean ± 95% CI across months)')

# Multiple lines by month
month_order = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
               'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
flights_pivot = flights.pivot(index='year', columns='month', values='passengers')[month_order]
flights_pivot[['Jan', 'Apr', 'Jul', 'Oct']].plot(ax=axes[1])
axes[1].set_title('Seasonal Growth by Quarter Start Month')
axes[1].set_ylabel('Passengers')

plt.tight_layout()
plt.show()

---
## 4 — Numerical × Categorical: Box and Violin Plots

**Reference:** [seaborn.boxplot](https://seaborn.pydata.org/generated/seaborn.boxplot.html) | [seaborn.stripplot](https://seaborn.pydata.org/generated/seaborn.stripplot.html)

These plots compare the distribution of a numerical variable across levels of a categorical variable.

> **Pro tip:** Overlay a `stripplot` on a `boxplot` to show individual data points. This is especially useful for smaller datasets.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Box + strip overlay
sns.boxplot(data=tips, x='day', y='total_bill', ax=axes[0],
            palette='Set2', order=['Thur', 'Fri', 'Sat', 'Sun'])
sns.stripplot(data=tips, x='day', y='total_bill', ax=axes[0],
              color='black', alpha=0.3, size=3,
              order=['Thur', 'Fri', 'Sat', 'Sun'])
axes[0].set_title('Bill by Day — Box + Strip overlay')

# Violin with hue
sns.violinplot(data=tips, x='day', y='tip', hue='sex',
               split=True, ax=axes[1], palette='Set1',
               order=['Thur', 'Fri', 'Sat', 'Sun'], inner='quart')
axes[1].set_title('Tip by Day × Sex — Split Violin')

plt.tight_layout()
plt.show()

---
## 5 — Numerical × Categorical: Bar Chart (Mean + CI)

**Reference:** [seaborn.barplot](https://seaborn.pydata.org/generated/seaborn.barplot.html)

Seaborn's `barplot` shows the **mean** of a numerical variable per category, with a **95% confidence interval** error bar.
This is more informative than a simple mean, as it shows estimation uncertainty.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Average tip by day
sns.barplot(data=tips, x='day', y='tip', ax=axes[0],
            palette='Set2', order=['Thur', 'Fri', 'Sat', 'Sun'],
            capsize=0.1)
axes[0].set_title('Mean Tip by Day (error bars = 95% CI)')

# Average tip by day × time
sns.barplot(data=tips, x='day', y='tip', hue='time', ax=axes[1],
            palette='Set1', order=['Thur', 'Fri', 'Sat', 'Sun'],
            capsize=0.1)
axes[1].set_title('Mean Tip by Day × Time (Lunch vs Dinner)')

plt.tight_layout()
plt.show()

---
## 6 — Categorical × Categorical: Grouped Bar and Heatmap

**Reference:** [pandas.crosstab](https://pandas.pydata.org/docs/reference/api/pandas.crosstab.html) | [seaborn.heatmap](https://seaborn.pydata.org/generated/seaborn.heatmap.html)

To see how two categorical variables relate:
- **Cross-tabulation (contingency table)** — count or percentage in each cell
- **Heatmap of the cross-tab** — visually highlights associations
- **Grouped bar chart** — compare counts or proportions side-by-side

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Cross-tab: class vs survived
ct = pd.crosstab(titanic['pclass'], titanic['survived'],
                  rownames=['Class'], colnames=['Survived'])
ct_pct = ct.div(ct.sum(axis=1), axis=0)  # row percentages
sns.heatmap(ct_pct, annot=True, fmt='.1%', cmap='RdYlGn',
            ax=axes[0], linewidths=0.5)
axes[0].set_title('Survival Rate by Passenger Class (%)')

# Grouped bar — embarked × sex
ct2 = pd.crosstab(titanic['embarked'], titanic['sex'])
ct2.plot(kind='bar', ax=axes[1], color=['salmon', 'steelblue'],
         rot=0, edgecolor='white')
axes[1].set_title('Embarkation Port by Sex')
axes[1].set_xlabel('Port')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

---
## Quick Reference — Bivariate Plot Selector

| Pair type | Recommended plot | When to use alternatives |
|---|---|---|
| Num × Num | Scatter | 2D-KDE if >1000 points; line if x is ordered |
| Num × Cat | Box + strip | Violin if shape matters; bar if comparing means |
| Cat × Cat | Grouped bar | Heatmap for many categories; stacked bar for proportions |

---
## Key Takeaways

- Always identify your pair type (Num×Num, Num×Cat, Cat×Cat) before choosing a plot
- Scatter plots reveal direction, strength, form, and outliers of numerical relationships
- Use 2D-KDE or hexbin when scatter is overplotted
- Bar charts with CI error bars communicate uncertainty — don't use bare means
- Cross-tab heatmaps reveal conditional distributions between categorical variables

---
## Further Reading

- [Kaggle — Data Visualization Course](https://www.kaggle.com/learn/data-visualization)
- [Seaborn — Relationship Plots](https://seaborn.pydata.org/tutorial/relational.html)
- [Seaborn — Categorical Plots](https://seaborn.pydata.org/tutorial/categorical.html)
- [Fundamentals of Data Visualization (free)](https://clauswilke.com/dataviz/) — Ch. 7-12
- [Storytelling with Data](https://www.storytellingwithdata.com/) — Cole Nussbaumer Knaflic